# 60 — SID Quantizer (W1) on Colab

Trains 3 RQ-VAE quantizers (seeds 42, 123, 7) over text + CF + audio embeddings, picks the best by cluster-purity, pins by SHA256. Output: `track_to_sid.parquet` on Drive.

**Wallclock**: ~10-15 min per seed on L4 (47K embeddings, batch=512, 50 epochs); ~30-45 min total for 3 seeds + pick-best.

**Spec**: `documents/specs/2026-05-15-sid-retrieval-design.md` §2.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth + Drive mount + symlink SID cache to Drive.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

DRIVE_SID_DIR = '/content/drive/MyDrive/recsys2026_sid_cache'
os.makedirs(DRIVE_SID_DIR, exist_ok=True)
REPO_SID_DIR = '/content/recsys2026/experiments/cache/sid'
os.makedirs(os.path.dirname(REPO_SID_DIR), exist_ok=True)
if os.path.lexists(REPO_SID_DIR):
    !rm -rf {REPO_SID_DIR}
!ln -s {DRIVE_SID_DIR} {REPO_SID_DIR}
print('symlinked', REPO_SID_DIR, '->', DRIVE_SID_DIR)

In [ ]:
# 4) Install deps.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml numpy scipy scikit-learn pyarrow vector-quantize-pytorch

In [ ]:
# 5) Smoke test: train one seed on first 500 tracks (~30 sec) -- verifies the pipeline before committing 30+ min.
!python scripts/build_sid_quantizer.py --seed 42 --epochs 5 --max-tracks 500

In [ ]:
# 5b) Inspect smoke output.
import json, os
smoke_gates = json.load(open('experiments/cache/sid/quantizer_seed42_gates.json'))
print(json.dumps(smoke_gates, indent=2))
print('all gates passed (on 500-track smoke):', smoke_gates['all_gates_passed'])

In [ ]:
# 6) Full run, all 3 seeds. ~10-15 min each on L4. Wipes the smoke artifacts first.
!rm -f experiments/cache/sid/quantizer_seed*
for seed in [42, 123, 7]:
    print(f'\n=== seed {seed} ===')
    !python scripts/build_sid_quantizer.py --seed {seed}

In [ ]:
# 7) Pick best of 3 + pin SHA256.
!python scripts/pick_best_sid_quantizer.py

In [ ]:
# 8) Final inspection -- what landed on Drive.
!ls -la experiments/cache/sid/
import pandas as pd
df = pd.read_parquet('experiments/cache/sid/track_to_sid.parquet')
print(f'\ntrack_to_sid.parquet: {len(df)} rows')
print(df.head(10))
n_unique = df.groupby(['code_1', 'code_2', 'code_3']).ngroups
n_collisions = (df.groupby(['code_1', 'code_2', 'code_3']).size() > 1).sum()
print(f'\nunique SIDs: {n_unique}, collision buckets: {n_collisions}')